In [3]:
# Cell 2: Imports and Data Prep
import json
import os
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load your movie dataset (e.g. TMDB 5000 or custom CSV)
# Replace 'movies.csv' with your dataset filename
df = pd.read_csv(r"C:\Users\esalo\Desktop\imarticus\Gemini_CLI\movie_recommendation\IMDB_Data.csv")

# Ensure necessary text columns have no NaN values
for col in ["title", "overview", "genres", "cast", "director", "keywords"]:
  if col in df.columns:
    df[col] = df[col].fillna("")
  else:
    df[col] = ""


# Combine metadata into an enriched semantic prompt for each movie
def build_movie_summary(row):
  return (
      f"Title: {row['title']}. "
      f"Genres: {row['genres']}. "
      f"Director: {row['director']}. "
      f"Cast: {row['cast']}. "
      f"Keywords: {row['keywords']}. "
      f"Overview: {row['overview']}"
  )


df["metadata_text"] = df.apply(build_movie_summary, axis=1)
print(f"Loaded {len(df)} movies.")
print("\nSample feature representation:")
print(df["metadata_text"].iloc[0][:300], "...")

Loaded 760803 movies.

Sample feature representation:
Title: . Genres: Romance. Director: . Cast: . Keywords: . Overview:  ...


In [4]:
# Cell 3: Dataset Quality and Column Inspection

print("1. Dataset Shape:")
print(df.shape)

print("\n2. Actual Column Names:")
print(df.columns.tolist())

print("\n3. First 5 Records:")
display(df.head())

print("\n4. Missing Values in Important Columns:")

for col in ["title", "overview", "genres", "cast", "director", "keywords"]:
    if col in df.columns:
        empty_count = (
            df[col].isna() | df[col].astype(str).str.strip().eq("")
        ).sum()

        percentage = empty_count / len(df) * 100 if len(df) else 0

        print(
            f"{col}: {empty_count:,} empty values "
            f"({percentage:.2f}%)"
        )
    else:
        print(f"{col}: COLUMN NOT FOUND")

print("\n5. First 10 Movie Titles:")

if "title" in df.columns:
    display(df[["title"]].head(10))
else:
    print("The expected 'title' column does not exist.")

print("\n6. Duplicate Records:")
print("Exact duplicate rows:", df.duplicated().sum())

1. Dataset Shape:
(760803, 15)

2. Actual Column Names:
['tconst', 'titleType', 'primaryTitle', 'startYear', 'genres', 'averageRating', 'numVotes', 'alternate_titles', 'cast_and_crew', 'title', 'overview', 'cast', 'director', 'keywords', 'metadata_text']

3. First 5 Records:


,tconst,titleType,primaryTitle,startYear,genres,averageRating,numVotes,alternate_titles,cast_and_crew,title,overview,cast,director,keywords,metadata_text
0,tt0000009,movie,Miss Jerry,1894,Romance,5.3,240.0,"Miss Jerry, Miss Jerry, Fräulein Jerry, Preili...","Blanche Bayliss, William Courtenay, Chauncey D...",,,,,,Title: . Genres: Romance. Director: . Cast: . ...
1,tt0000147,movie,The Corbett-Fitzsimmons Fight,1897,"Documentary,News,Sport",5.3,618.0,"The Corbett-Fitzsimmons Fight, Corbeti ja Fitz...",Enoch J. Rector,,,,,,"Title: . Genres: Documentary,News,Sport. Direc..."
2,tt0000502,movie,Bohemios,1905,\N,3.7,27.0,"Bohemios, Bohemios","Antonio del Pozo, El Mochuelo, Ricardo de Baños",,,,,,Title: . Genres: \N. Director: . Cast: . Keywo...
3,tt0000574,movie,The Story of the Kelly Gang,1906,"Action,Adventure,Biography",6.0,1102.0,"The Story of the Kelly Gang, The Story of the ...","Elizabeth Tait, John Tait, Nicholas Brierley, ...",,,,,,"Title: . Genres: Action,Adventure,Biography. D..."
4,tt0000591,movie,The Prodigal Son,1907,Drama,5.1,43.0,"L'enfant prodigue, The Prodigal Son, The Prodi...","Georges Wague, Henri Gouget, Christiane Mandel...",,,,,,Title: . Genres: Drama. Director: . Cast: . Ke...



4. Missing Values in Important Columns:
title: 760,803 empty values (100.00%)
overview: 760,803 empty values (100.00%)
genres: 0 empty values (0.00%)
cast: 760,803 empty values (100.00%)
director: 760,803 empty values (100.00%)
keywords: 760,803 empty values (100.00%)

5. First 10 Movie Titles:


,title
0,
1,
2,
3,
4,
5,
6,
7,
8,
9,



6. Duplicate Records:
Exact duplicate rows: 0


In [4]:
# Cell 3: Generate Dense Vector Embeddings
# 'all-MiniLM-L6-v2' is fast, lightweight, and runs on CPU inside 8GB RAM
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Encoding movie metadata into vector space...")
movie_embeddings = embedding_model.encode(
    df["metadata_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

# Normalize vectors so inner product equals cosine similarity
faiss.normalize_L2(movie_embeddings)
print("Embedding Shape:", movie_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding movie metadata into vector space...


Batches:   0%|          | 0/11888 [00:00<?, ?it/s]

Embedding Shape: (760803, 384)


In [5]:
# Cell 4: Build FAISS Vector Index
dimension = movie_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner Product on normalized vectors = Cosine Similarity
index.add(movie_embeddings)

print(f"FAISS index built successfully with {index.ntotal} vectors.")

FAISS index built successfully with 760803 vectors.


In [6]:
# Cell 5: Search & Retrieval Function
def get_recommendations(
    query, top_k=5, filter_genre=None, filter_actor=None
):
  # Check if query matches an exact title
  title_match = df[df["title"].str.lower() == query.lower()]

  if not title_match.empty:
    idx = title_match.index[0]
    query_vector = movie_embeddings[idx : idx + 1]
    # We query top_k + 1 to exclude the input movie itself
    k_search = top_k + 1
  else:
    # Natural language query (actor, theme, emotion, plot)
    query_vector = embedding_model.encode([query])
    faiss.normalize_L2(query_vector)
    k_search = top_k

  # Retrieve top vectors
  scores, indices = index.search(query_vector, k_search)

  results = []
  for score, idx in zip(scores[0], indices[0]):
    row = df.iloc[idx]
    # Skip identical title if user queried an exact movie
    if not title_match.empty and row["title"].lower() == query.lower():
      continue

    # Apply optional hard filters (Genre / Actor)
    if filter_genre and filter_genre.lower() not in row["genres"].lower():
      continue
    if filter_actor and filter_actor.lower() not in row["cast"].lower():
      continue

    results.append({
        "title": row["title"],
        "similarity_score": round(float(score) * 100, 2),
        "genres": row["genres"],
        "director": row["director"],
        "cast": row["cast"],
        "overview": row["overview"],
    })
    if len(results) == top_k:
      break

  return pd.DataFrame(results)

In [8]:
import os
from google import genai

# Uncomment and insert your actual key
os.environ["GEMINI_API_KEY"] = "AQ.Ab8RN6J4l3pNunXh06M-dnknBfhbO_ug9MpT9urUQ9If8SzLMg"

# The client automatically picks up GEMINI_API_KEY
client = genai.Client()

In [13]:
# Cell 9: Updated LLM Function
def generate_llm_insights(user_query, recommended_df):
  # Prepare context from retrieved movies
  context_items = []
  for _, row in recommended_df.iterrows():
    context_items.append(
        f"- {row['title']} ({row['similarity_score']}% Match): "
        f"Directed by {row['director']}, Starring: {row['cast']}. "
        f"Plot: {row['overview'][:180]}..."
    )
  context_text = "\n".join(context_items)

  prompt = f"""
    You are an intelligent streaming platform recommender system (like Netflix/Hotstar).
    The user asked: "{user_query}"
    
    Our vector retrieval engine selected these candidate movies:
    {context_text}
    
    Task:
    1. Write a short, engaging 2-sentence hook matching the user's vibe.
    2. For each movie, provide a 1-sentence "Why Recommended" reason pointing out specific connections (director style, plot twists, actor chemistry).
    3. Add a quick 3-word mood tag for each (e.g., [Dark, Gritty, Thriller]).
    """

  # Change model to "gemini-3.8-flash"
  response = client.models.generate_content(
      model="gemini-3.8-flash", contents=prompt
  )
  return response.text

In [14]:
# Cell 7: Execution Example
query = "Fast-paced space exploration with intense emotional soundtrack"
recs = get_recommendations(query=query, top_k=3)

print("--- RETRIEVED MOVIES ---")
display(recs[["title", "similarity_score", "genres", "director"]])

print("\n--- LLM RECOMMENDATION INSIGHTS ---")
insights = generate_llm_insights(query, recs)
print(insights)

--- RETRIEVED MOVIES ---


,title,similarity_score,genres,director
0,,50.27,"Adventure,Musical,Sci-Fi",
1,,50.27,"Adventure,Musical,Sci-Fi",
2,,50.27,"Adventure,Musical,Sci-Fi",



--- LLM RECOMMENDATION INSIGHTS ---


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x00000250384A5160>>
Traceback (most recent call last):
  File "C:\Users\esalo\anaconda3\Lib\site-packages\ipykernel\ipkernel.py", line 790, in _clean_thread_parent_frames
    active_threads = {thread.ident for thread in threading.enumerate()}
  File "C:\Users\esalo\anaconda3\Lib\threading.py", line 1479, in enumerate
    def enumerate():
KeyboardInterrupt: 


KeyboardInterrupt: 

In [15]:
# Cell 8: Test with Actor + Vibe query
query = "Christopher Nolan psychological mind twists"
recs_nolan = get_recommendations(query=query, top_k=4)
display(recs_nolan[["title", "similarity_score", "genres", "cast"]])

,title,similarity_score,genres,cast
0,,33.92,"Fantasy,Horror,Sci-Fi",
1,,33.92,"Fantasy,Horror,Sci-Fi",
2,,33.92,"Fantasy,Horror,Sci-Fi",
3,,33.92,"Fantasy,Horror,Sci-Fi",


In [16]:
import requests

TMDB_API_KEY = "8af612e058ec64c6d0812df056e81b56"  # Replace with your TMDB v3 API key
IMAGE_BASE_URL = "https://image.tmdb.org/t/p/w500"  # w500 gives clean poster resolution


def fetch_poster_by_id(movie_id):
  """Fetches poster URL directly using the TMDB movie ID."""
  url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={TMDB_API_KEY}&language=en-US"
  try:
    response = requests.get(url, timeout=5)
    data = response.json()
    poster_path = data.get("poster_path")
    if poster_path:
      return f"{IMAGE_BASE_URL}{poster_path}"
  except Exception:
    pass
  return "https://via.placeholder.com/500x750?text=No+Poster+Available"


def fetch_poster_by_title(title):
  """Fallback: searches TMDB by movie title if ID is not available."""
  search_url = f"https://api.themoviedb.org/3/search/movie?api_key={TMDB_API_KEY}&query={requests.utils.quote(title)}"
  try:
    response = requests.get(search_url, timeout=5)
    data = response.json()
    results = data.get("results", [])
    if results and results[0].get("poster_path"):
      return f"{IMAGE_BASE_URL}{results[0]['poster_path']}"
  except Exception:
    pass
  return "https://via.placeholder.com/500x750?text=No+Poster+Available"

In [17]:
from IPython.display import HTML, display


def display_recommendations_with_posters(recs_df):
  html_content = """
    <div style="display: flex; flex-wrap: wrap; gap: 20px; justify-content: flex-start; background: #0f1016; padding: 20px; border-radius: 12px; font-family: sans-serif;">
    """

  for _, row in recs_df.iterrows():
    # If your dataset has an 'id' column, use fetch_poster_by_id(row['id'])
    # Otherwise use fetch_poster_by_title(row['title'])
    if "id" in row and pd.notnull(row["id"]):
      poster_url = fetch_poster_by_id(row["id"])
    else:
      poster_url = fetch_poster_by_title(row["title"])

    html_content += f"""
        <div style="width: 180px; background: #1a1c24; border-radius: 10px; overflow: hidden; box-shadow: 0 4px 12px rgba(0,0,0,0.5); color: #fff;">
            <img src="{poster_url}" style="width: 100%; height: 260px; object-fit: cover;" />
            <div style="padding: 10px;">
                <span style="background: #00d26a; color: #000; font-weight: bold; font-size: 11px; padding: 2px 6px; border-radius: 4px;">
                    {row['similarity_score']}% Match
                </span>
                <h4 style="margin: 8px 0 4px 0; font-size: 14px; white-space: nowrap; overflow: hidden; text-overflow: ellipsis;" title="{row['title']}">
                    {row['title']}
                </h4>
                <p style="margin: 0; font-size: 12px; color: #9da3af;">{row['genres'][:25]}...</p>
            </div>
        </div>
        """

  html_content += "</div>"
  display(HTML(html_content))

In [18]:
# 1. Get recommendations using your vector search
query = "Christopher Nolan psychological mind twists"
recs = get_recommendations(query=query, top_k=5)

# 2. Render the interactive posters
display_recommendations_with_posters(recs)

# 3. Generate Gemini explanations
insights = generate_llm_insights(query, recs)
print("\n--- LLM RECOMMENDATION INSIGHTS ---")
print(insights)

KeyboardInterrupt: 

In [19]:
cols = st.columns(len(recs))

for idx, col in enumerate(cols):
  row = recs.iloc[idx]
  poster_url = fetch_poster_by_title(row["title"])

  with col:
    st.image(poster_url, use_container_width=True)
    st.markdown(f"**{row['title']}**")
    st.caption(f"⭐ {row['similarity_score']}% Match")

NameError: name 'st' is not defined

In [21]:
import requests

# Test your TMDB key directly:
TEST_API_KEY = "YOUR_TMDB_API_KEY"  # <-- paste your TMDB key here
test_url = f"https://api.themoviedb.org/3/search/movie?api_key={TEST_API_KEY}&query=Inception"

res = requests.get(test_url)
print("Status Code:", res.status_code)
if res.status_code == 200:
  data = res.json()
  if data.get("results"):
    poster = data["results"][0].get("poster_path")
    print(
        "Poster URL Found:", f"https://image.tmdb.org/t/p/w500{poster}"
    )  #
  else:
    print("Movie search worked, but no results found.")
else:
  print("TMDB Error Response:", res.text)

ConnectTimeout: HTTPSConnectionPool(host='api.themoviedb.org', port=443): Max retries exceeded with url: /3/search/movie?api_key=YOUR_TMDB_API_KEY&query=Inception (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000002511D0AE990>, 'Connection to api.themoviedb.org timed out. (connect timeout=None)'))

In [22]:
import requests

OMDB_API_KEY = "991878c4"  # <-- paste your 8-character OMDb key here
url = f"https://www.omdbapi.com/?t=Inception&apikey={OMDB_API_KEY}"

try:
  res = requests.get(url, timeout=5)
  data = res.json()
  print("Status:", res.status_code)
  print("Movie Found:", data.get("Title"))
  print("Poster URL:", data.get("Poster"))
except Exception as e:
  print("Error:", e)

Status: 200
Movie Found: Inception
Poster URL: https://m.media-amazon.com/images/M/MV5BMjAxMzY3NjcxNF5BMl5BanBnXkFtZTcwNTI5OTM0Mw@@._V1_QL75_UX380_CR0,0,380,562_.jpg


In [23]:
import html
import urllib.parse
from IPython.display import HTML, display


def generate_local_poster(title, genre=""):
  # Generates an inline SVG data URI poster so no external network call is needed
  safe_title = html.escape(str(title)[:25])
  safe_genre = html.escape(str(genre)[:20])

  svg = f"""<svg xmlns="http://www.w3.org/2000/svg" width="300" height="450" viewBox="0 0 300 450">
      <defs>
        <linearGradient id="g" x1="0%" y1="0%" x2="100%" y2="100%">
          <stop offset="0%" stop-color="#1e1b4b"/>
          <stop offset="50%" stop-color="#0f172a"/>
          <stop offset="100%" stop-color="#020617"/>
        </linearGradient>
      </defs>
      <rect width="100%" height="100%" fill="url(#g)"/>
      <circle cx="150" cy="180" r="50" fill="#312e81" opacity="0.5"/>
      <polygon points="142,160 168,180 142,200" fill="#38bdf8"/>
      <text x="50%" y="280" font-family="system-ui, sans-serif" font-size="18" font-weight="700" fill="#f8fafc" text-anchor="middle">{safe_title}</text>
      <text x="50%" y="310" font-family="system-ui, sans-serif" font-size="12" fill="#94a3b8" text-anchor="middle">{safe_genre}</text>
      <text x="50%" y="410" font-family="system-ui, sans-serif" font-size="10" letter-spacing="2" fill="#475569" text-anchor="middle">CINEMA AI</text>
    </svg>"""

  encoded = urllib.parse.quote(svg)
  return f"data:image/svg+xml;utf8,{encoded}"


def display_recommendations_offline(recs_df):
  cards_html = []
  for _, row in recs_df.iterrows():
    title = row.get("title", "Unknown Title")
    score = row.get("similarity_score", 0)
    genres = str(row.get("genres", ""))[:28]

    poster_src = generate_local_poster(title, genres)

    card = f"""
        <div style="flex: 0 0 170px; background-color: #111827; border-radius: 12px; overflow: hidden; margin: 8px; box-shadow: 0 8px 16px rgba(0,0,0,0.6); font-family: system-ui, sans-serif; border: 1px solid #1f2937;">
            <div style="width: 100%; height: 250px;">
                <img src="{poster_src}" style="width: 100%; height: 100%; object-fit: cover; display: block;" />
            </div>
            <div style="padding: 10px;">
                <span style="background-color: #10b981; color: #022c22; font-weight: 700; font-size: 11px; padding: 2px 7px; border-radius: 4px; display: inline-block;">
                    {score}% Match
                </span>
                <div style="color: #f9fafb; font-weight: 600; font-size: 13px; margin-top: 6px; line-height: 1.2; height: 32px; overflow: hidden;" title="{title}">
                    {title}
                </div>
                <div style="color: #9ca3af; font-size: 11px; margin-top: 4px; overflow: hidden; text-overflow: ellipsis; white-space: nowrap;">
                    {genres}
                </div>
            </div>
        </div>
        """
    cards_html.append(card)

  full_html = f"""
    <div style="display: flex; flex-wrap: wrap; background-color: #030712; padding: 16px; border-radius: 16px; width: 100%;">
        {''.join(cards_html)}
    </div>
    """
  display(HTML(full_html))

In [24]:
query = "Interstellar"
recs = get_recommendations(query=query, top_k=4)
display_recommendations_offline(recs)

In [25]:
# In your notebook
import pickle

# Save the FAISS index
faiss.write_index(index, "movie_vector_index.faiss")

# Save the DataFrame with essential fields
df[["title", "genres", "cast", "director", "overview"]].to_pickle(
    "movies_processed.pkl"
)